# Tier 5 Solutions — Set 1: Modern AI for Science

Solutions for Assignment Set 1: Modern AI for Science.

## Complicated moments explained
These are common points where learners usually get stuck:
- Model score != biological truth. Treat predictions as ranked hypotheses requiring validation.
- Context length and tokenization can change model behavior more than minor hyperparameter tweaks.
- Domain shift is common: performance on curated benchmarks may not transfer to your assay/data source.


In [ ]:
import numpy as np
import math
from math import log, sqrt, cos, pi
import random

## Problem 1: LoRA Rank Decomposition (1 star, 10 pts)

In [ ]:
def lora_decompose(W: np.ndarray, r: int) -> dict:
    """
    Approximate weight matrix W with a rank-r LoRA decomposition.

    Args:
        W: Weight matrix of shape (d_out, d_in).
        r: LoRA rank (must be <= min(d_out, d_in)).

    Returns:
        Dict with keys 'B' (d_out, r), 'A' (r, d_in), 'reconstruction_error' (float).
    """
    U, S, Vt = np.linalg.svd(W, full_matrices=False)

    # Keep top-r singular values
    B = U[:, :r] * np.sqrt(S[:r])
    A = np.diag(np.sqrt(S[:r])) @ Vt[:r, :]

    reconstruction_error = float(np.linalg.norm(W - B @ A, 'fro'))

    return {"B": B, "A": A, "reconstruction_error": reconstruction_error}


rng = np.random.default_rng(42)
W_test = rng.standard_normal((8, 6))
result = lora_decompose(W_test, r=2)
print(f"B shape: {result['B'].shape}")
print(f"A shape: {result['A'].shape}")
print(f"Reconstruction error (r=2): {result['reconstruction_error']:.4f}")

# Full rank should give near-zero error
result_full = lora_decompose(W_test, r=6)
print(f"Reconstruction error (r=6, full): {result_full['reconstruction_error']:.6f}")

## Problem 2: LoRA Parameter Efficiency (1 star, 10 pts)

In [ ]:
def lora_efficiency(d_model: int, num_layers: int, lora_rank: int) -> dict:
    """
    Compute LoRA parameter efficiency compared to full fine-tuning of attention weights.

    Args:
        d_model: Hidden size (dimension of each attention matrix side).
        num_layers: Number of transformer layers.
        lora_rank: LoRA rank r.

    Returns:
        Dict with 'original_params', 'lora_params', 'compression_ratio'.
    """
    # 4 attention matrices (Q, K, V, O) each d_model x d_model
    original_params = 4 * d_model * d_model * num_layers

    # LoRA adapts Q and V only; each gets A (r x d_model) and B (d_model x r)
    lora_params = 2 * 2 * lora_rank * d_model * num_layers

    compression_ratio = lora_params / original_params

    return {
        "original_params":   original_params,
        "lora_params":       lora_params,
        "compression_ratio": compression_ratio,
    }


# LLaMA-7B-style model
result = lora_efficiency(d_model=4096, num_layers=32, lora_rank=8)
print(f"Original attention params : {result['original_params']:,}")
print(f"LoRA params               : {result['lora_params']:,}")
print(f"Compression ratio         : {result['compression_ratio']:.4f}")
print(f"Parameter reduction       : {1/result['compression_ratio']:.1f}x")

## Problem 3: Chat Template Formatting (2 stars, 10 pts)

In [ ]:
def format_llama3_chat(messages: list[dict]) -> str:
    """
    Format a list of messages using the LLaMA-3 chat template.

    Args:
        messages: List of dicts with 'role' ('system', 'user', 'assistant')
                  and 'content' (str).

    Returns:
        Formatted prompt string ready for model input.
    """
    result = ""
    first_non_system = True

    for i, msg in enumerate(messages):
        role    = msg['role']
        content = msg['content']

        if role == 'system':
            result += f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{content}<|eot_id|>"
            first_non_system = False
        elif role == 'user':
            if first_non_system:
                result += "<|begin_of_text|>"
                first_non_system = False
            result += f"<|start_header_id|>user<|end_header_id|>\n\n{content}<|eot_id|>"
        elif role == 'assistant':
            result += f"<|start_header_id|>assistant<|end_header_id|>\n\n{content}<|eot_id|>"

    # Always append the assistant header to prompt a response
    result += "<|start_header_id|>assistant<|end_header_id|>\n\n"

    return result


messages = [
    {"role": "system", "content": "You are a helpful bioinformatics assistant."},
    {"role": "user", "content": "What is a FASTA file?"},
    {"role": "assistant", "content": "A FASTA file stores biological sequences with a header line starting with >."},
    {"role": "user", "content": "How do I parse one in Python?"},
]

formatted = format_llama3_chat(messages)
print(repr(formatted))
print()
print(formatted)

## Problem 4: TF-IDF Document Retrieval (2 stars, 15 pts)

In [ ]:
def tfidf_retrieve(corpus: list[str], query: str, k: int = 3) -> list[int]:
    """
    Retrieve the top-k most relevant document indices using TF-IDF scoring.

    Args:
        corpus: List of document strings.
        query: Query string.
        k: Number of top documents to return.

    Returns:
        List of up to k document indices sorted by descending TF-IDF score.
    """
    N = len(corpus)

    # Tokenize
    def tokenize(text):
        return text.lower().split()

    query_terms = tokenize(query)

    # Compute document frequency for each term
    df = {}
    tokenized_corpus = [tokenize(doc) for doc in corpus]
    for tokens in tokenized_corpus:
        for term in set(tokens):
            df[term] = df.get(term, 0) + 1

    # Score each document
    scores = []
    for doc_tokens in tokenized_corpus:
        total_terms = len(doc_tokens)
        score = 0.0
        term_count = {}
        for t in doc_tokens:
            term_count[t] = term_count.get(t, 0) + 1

        for term in query_terms:
            tf  = term_count.get(term, 0) / total_terms if total_terms > 0 else 0.0
            idf = log((1 + N) / (1 + df.get(term, 0))) + 1
            score += tf * idf

        scores.append(score)

    # Return top-k indices by descending score
    ranked = sorted(range(N), key=lambda i: scores[i], reverse=True)
    return ranked[:k]


corpus = [
    "CRISPR is a gene editing tool derived from bacterial immune systems.",
    "RNA sequencing measures gene expression levels across the transcriptome.",
    "CRISPR Cas9 enables precise gene editing by cutting DNA at target sites.",
    "Single cell RNA sequencing reveals cell type heterogeneity in tissues.",
    "Protein folding determines the three-dimensional structure of proteins.",
    "AlphaFold predicts protein structure from amino acid sequence using deep learning.",
]

query = "CRISPR gene editing"
top_k = tfidf_retrieve(corpus, query, k=3)
print(f"Query: {query!r}")
print(f"Top-{len(top_k)} results:")
for idx in top_k:
    print(f"  [{idx}] {corpus[idx]}")

## Problem 5: DDPM Forward Noise Process (1 star, 15 pts)

In [ ]:
def ddpm_forward(x0: np.ndarray, betas: list[float], t: int, seed: int = 0) -> dict:
    """
    Apply the DDPM forward diffusion process to get the noisy sample at timestep t.

    Args:
        x0: Clean sample, 1D numpy array.
        betas: Noise schedule, list of T beta values (0-indexed, beta[0] = beta_1).
        t: Timestep (1-indexed, so t=1 uses betas[0]).
        seed: Random seed for reproducibility.

    Returns:
        Dict with 'x_t', 'alpha_bar_t', 'noise'.
    """
    # Cumulative product of (1 - beta) for timesteps 1..t
    alpha_bar_t = 1.0
    for i in range(t):
        alpha_bar_t *= (1.0 - betas[i])

    np.random.seed(seed)
    eps = np.random.randn(*x0.shape)

    x_t = sqrt(alpha_bar_t) * x0 + sqrt(1.0 - alpha_bar_t) * eps

    return {"x_t": x_t, "alpha_bar_t": alpha_bar_t, "noise": eps}


x0 = np.array([1.0, 0.5, -0.3, 0.8])
T = 10
betas = [0.001 + 0.001 * i for i in range(T)]  # simple linear schedule

for t in [1, 5, 10]:
    res = ddpm_forward(x0, betas, t, seed=42)
    print(f"t={t:2d}: alpha_bar={res['alpha_bar_t']:.4f}  x_t={np.round(res['x_t'], 4)}")

## Problem 6: Linear and Cosine Noise Schedules (2 stars, 15 pts)

In [ ]:
def noise_schedules(T: int, beta_start: float = 1e-4, beta_end: float = 0.02) -> dict:
    """
    Compute linear and cosine noise schedules.

    Args:
        T: Number of diffusion timesteps.
        beta_start: Starting beta for linear schedule.
        beta_end: Ending beta for linear schedule.

    Returns:
        Dict with 'linear_betas', 'cosine_betas', 'linear_alpha_bars', 'cosine_alpha_bars'.
    """
    # Linear schedule
    linear_betas = [beta_start + (beta_end - beta_start) * t / (T - 1) for t in range(T)]

    # Linear alpha_bars: cumprod of (1 - beta)
    linear_alpha_bars = []
    prod = 1.0
    for b in linear_betas:
        prod *= (1.0 - b)
        linear_alpha_bars.append(prod)

    # Cosine schedule
    s = 0.008
    f0 = cos(((0 / T + s) / (1 + s)) * pi / 2) ** 2

    alpha_bar_cosine = []
    for t in range(T):
        ft = cos(((t / T + s) / (1 + s)) * pi / 2) ** 2
        alpha_bar_cosine.append(ft / f0)

    cosine_betas = []
    for t in range(T):
        if t == 0:
            # alpha_bar_{-1} = 1
            b = 1.0 - alpha_bar_cosine[0]
        else:
            b = min(1.0 - alpha_bar_cosine[t] / alpha_bar_cosine[t-1], 0.999)
        cosine_betas.append(b)

    return {
        "linear_betas":       linear_betas,
        "cosine_betas":       cosine_betas,
        "linear_alpha_bars":  linear_alpha_bars,
        "cosine_alpha_bars":  alpha_bar_cosine,
    }


schedules = noise_schedules(T=1000)
print(f"Linear betas  — first: {schedules['linear_betas'][0]:.6f}  last: {schedules['linear_betas'][-1]:.6f}")
print(f"Cosine betas  — first: {schedules['cosine_betas'][0]:.6f}  last: {schedules['cosine_betas'][-1]:.6f}")
print(f"Linear alpha_bars — first: {schedules['linear_alpha_bars'][0]:.6f}  last: {schedules['linear_alpha_bars'][-1]:.6f}")
print(f"Cosine alpha_bars — first: {schedules['cosine_alpha_bars'][0]:.6f}  last: {schedules['cosine_alpha_bars'][-1]:.6f}")

## Problem 7: Cosine Similarity for Multi-Vector Retrieval (3 stars, 10 pts)

In [ ]:
def maxsim(Q: np.ndarray, D: np.ndarray) -> float:
    """
    Compute the MaxSim score between a query and a document using multi-vector representations.

    Args:
        Q: Query token embeddings, shape (q_tokens, dim).
        D: Document token embeddings, shape (d_tokens, dim).

    Returns:
        MaxSim score as a float.
    """
    q_norms = np.linalg.norm(Q, axis=1, keepdims=True)
    d_norms = np.linalg.norm(D, axis=1, keepdims=True)

    # Avoid division by zero
    q_norms = np.where(q_norms == 0, 1e-10, q_norms)
    d_norms = np.where(d_norms == 0, 1e-10, d_norms)

    Q_norm = Q / q_norms
    D_norm = D / d_norms

    # Cosine similarity matrix: (q_tokens, d_tokens)
    sim_matrix = Q_norm @ D_norm.T

    # MaxSim: for each query token, take max over document tokens; then average
    max_sims = sim_matrix.max(axis=1)
    return float(max_sims.mean())


rng = np.random.default_rng(7)
Q = rng.standard_normal((4, 8))   # 4 query tokens, dim=8
D = rng.standard_normal((10, 8))  # 10 document tokens, dim=8

score = maxsim(Q, D)
print(f"MaxSim score: {score:.4f}")

# Identical query and document (each row is the same) should give score ~ 1.0
identical = rng.standard_normal((1, 8))
same_score = maxsim(np.tile(identical, (3, 1)), np.tile(identical, (3, 1)))
print(f"MaxSim (identical Q=D): {same_score:.4f}  (expected ~1.0)")

## Problem 8: DDIM Deterministic Sampling Step (3 stars, 15 pts)

In [ ]:
def ddim_step(
    x_t: np.ndarray,
    predicted_noise: np.ndarray,
    t: int,
    t_prev: int,
    alpha_bars: list[float],
) -> dict:
    """
    Perform one deterministic DDIM denoising step.

    Args:
        x_t: Current noisy sample at timestep t.
        predicted_noise: Noise predicted by the model, same shape as x_t.
        t: Current timestep index (used to index alpha_bars).
        t_prev: Previous (less noisy) timestep index (t_prev < t).
        alpha_bars: List of cumulative alpha_bar values for all timesteps.

    Returns:
        Dict with 'x_t_prev' and 'predicted_x0', both numpy arrays.
    """
    alpha_bar_t      = alpha_bars[t]
    alpha_bar_t_prev = alpha_bars[t_prev]

    # Predict clean sample x0
    predicted_x0 = (x_t - sqrt(1.0 - alpha_bar_t) * predicted_noise) / sqrt(alpha_bar_t)

    # Direction pointing to x_t
    direction = sqrt(1.0 - alpha_bar_t_prev) * predicted_noise

    # DDIM update (deterministic, no stochastic term)
    x_t_prev = sqrt(alpha_bar_t_prev) * predicted_x0 + direction

    return {"x_t_prev": x_t_prev, "predicted_x0": predicted_x0}


# Build a simple cosine schedule for testing
T = 100
s = 0.008
alpha_bars_test = [
    (cos(((t / T + s) / (1 + s)) * pi / 2) ** 2) / (cos((s / (1 + s)) * pi / 2) ** 2)
    for t in range(T)
]

rng = np.random.default_rng(3)
x_t = rng.standard_normal(4)
eps = rng.standard_normal(4)

result = ddim_step(x_t, eps, t=80, t_prev=60, alpha_bars=alpha_bars_test)
print(f"x_t         : {np.round(x_t, 4)}")
print(f"predicted_x0: {np.round(result['predicted_x0'], 4)}")
print(f"x_t_prev    : {np.round(result['x_t_prev'], 4)}")